# Clase 14 — Web Scraping con BeautifulSoup

## Objetivos de la clase

Al terminar esta clase vas a poder:

- Explicar qué es el web scraping y por qué se usa cuando no existe una API.
- Convertir un bloque de HTML en un objeto navegable con `BeautifulSoup` y visualizarlo con `prettify()`.
- Entender qué es un `Tag`, acceder a sus **atributos** y a su contenido de texto (`NavigableString`).
- Navegar el árbol HTML con `parent`, `next_sibling` y accesos directos como `soup.h3`.
- Usar `find()` y `find_all()` con distintos filtros (nombre, atributos, `class_`, texto) para localizar elementos.
- Extraer una tabla HTML "a mano", recorriendo filas (`find_all('tr')`) y celdas (`find_all('td')`).
- Extraer la misma tabla con el atajo de pandas `pd.read_html()`.

> Requisito: ya conoces Python (funciones, bucles, diccionarios), pandas básico y cómo pedir datos con `requests` a una API. No hace falta conexión a internet para esta clase — todo el HTML que usamos está definido como texto en el propio notebook.

## ¿Por qué nos importa esto?

Cuando una API existe, siempre es la mejor opción para conseguir datos: te da información limpia y estructurada. El problema es que la mayoría de los datos públicos en internet **no tienen una API detrás** — están incrustados en el HTML de una página, pensados para que los lea una persona en un navegador, no una máquina.

El **web scraping** es la técnica de tomar ese HTML y extraer de ahí los datos que te interesan, automatizando lo que harías a mano con copiar y pegar. `BeautifulSoup` es la librería de Python que convierte el HTML (un bloque de texto) en una estructura navegable de objetos, para que puedas recorrerla y filtrarla con código en vez de leerla vos mismo.

**Nota sobre la instalación:** `BeautifulSoup` vive en el paquete `bs4`, que puede no venir preinstalado en un Jupyter local (a diferencia de Google Colab, donde ya está). Si al ejecutar `import bs4` te da un `ModuleNotFoundError`, instalalo con:

```
pip install beautifulsoup4
```

Más abajo hay una celda opcional con `!pip install beautifulsoup4` por si preferís ejecutarlo directamente desde el notebook.

**Celda opcional** — solo ejecutala si `import bs4` te falla más abajo. No hace falta correrla si `bs4` ya está instalado en tu entorno.

In [ ]:
# Celda OPCIONAL: descomenta la siguiente línea solo si no tenés bs4 instalado
# !pip install beautifulsoup4

## Nuestro HTML de ejemplo: una tabla de productos

En vez de descargar una página real con `requests` (que puede no estar disponible sin conexión, o cambiar con el tiempo), vamos a trabajar sobre un HTML de ejemplo definido directamente como **string de Python**. Así el notebook funciona igual con o sin internet.

Definimos una tabla simple con 4 productos ficticios: nombre, categoría, precio y stock.

In [ ]:
html_productos = """
<!DOCTYPE html>
<html>
<head><title>Catálogo de productos</title></head>
<body>
<h3 id="titulo_tabla">Catálogo de la tienda</h3>
<p>Estos son nuestros <b id="destacado">productos destacados</b> de esta semana.</p>
<table id="tabla_productos" class="catalogo">
  <tr>
    <th>Nombre</th>
    <th>Categoría</th>
    <th>Precio</th>
    <th>Stock</th>
  </tr>
  <tr>
    <td>Mouse inalámbrico</td>
    <td>Accesorios</td>
    <td>$15.99</td>
    <td>120</td>
  </tr>
  <tr>
    <td>Teclado mecánico</td>
    <td>Accesorios</td>
    <td>$49.50</td>
    <td>45</td>
  </tr>
  <tr>
    <td>Monitor 24 pulgadas</td>
    <td><a href="https://ejemplo.com/categoria/pantallas">Pantallas</a></td>
    <td>$189.00</td>
    <td>10</td>
  </tr>
  <tr>
    <td>Auriculares Bluetooth</td>
    <td>Audio</td>
    <td>$34.75</td>
    <td>0</td>
  </tr>
</table>
</body>
</html>
"""

print(html_productos[:120], "...")

## Creando el objeto `BeautifulSoup`

Para parsear el HTML lo pasamos al constructor `BeautifulSoup(html, parser)`. El segundo argumento indica qué "motor" de parsing usar; `'html.parser'` viene incluido en la librería estándar de Python, así que no necesita instalación adicional (a diferencia de `'html5lib'` o `'lxml'`, que son alternativas más tolerantes a HTML mal formado pero requieren instalarse aparte).

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_productos, 'html.parser')
print(type(soup))

Con `prettify()` podés imprimir el HTML reindentado según su jerarquía de tags — muy útil para entender visualmente cómo está anidada una página antes de escribir el código que la recorre.

In [ ]:
print(soup.prettify())

## Tags: acceder a un elemento del HTML

Cuando accedés a `soup.<nombre_de_tag>`, BeautifulSoup te devuelve un objeto `Tag` — la representación de ese elemento. Si hay varios tags con el mismo nombre, te devuelve **el primero** que encuentra en el documento.

In [ ]:
tag_titulo = soup.title
print("tag titulo:", tag_titulo)
print("tipo:", type(tag_titulo))

tag_h3 = soup.h3   # el primer <h3> de todo el documento
print("tag h3:", tag_h3)

## Navegación: children, parent y siblings

Desde cualquier `Tag` te podés mover en las tres direcciones del árbol HTML:

- **Hijo** (*child*): accediendo con `.nombre_del_tag` bajás un nivel.
- **Padre** (*parent*): con `.parent` subís un nivel.
- **Hermano** (*sibling*): con `.next_sibling` te movés al elemento que sigue al mismo nivel.

Un detalle importante con `.next_sibling`: cuando el HTML original tiene saltos de línea o espacios entre un tag y el siguiente (como en nuestro `html_productos`, escrito en varias líneas para que se lea mejor), ese espacio en blanco también cuenta como un "hermano" — un `NavigableString` vacío, no el tag que buscabas. Para saltar directamente al **próximo tag** (ignorando espacios en blanco sueltos), usá `.find_next_sibling()` en su lugar.

In [ ]:
tag_p = soup.p            # el primer <p> del documento
tag_child = tag_p.b       # bajamos un nivel: el <b> que está adentro del <p>
print("hijo (b):", tag_child)

tag_parent = tag_child.parent   # subimos: volvemos al <p> que lo contiene
print("padre:", tag_parent)

# next_sibling, tal cual, puede devolver un salto de línea en vez del tag esperado
print("next_sibling 'crudo':", repr(soup.h3.next_sibling))

# find_next_sibling() salta los espacios en blanco y devuelve directamente el próximo tag
siguiente = soup.h3.find_next_sibling()   # el <p> que sigue al <h3>
print("siguiente hermano de h3:", siguiente)

## Atributos y el contenido de texto

Un tag con atributos se comporta como un diccionario de Python: podés acceder al valor de un atributo con corchetes, con `.get()`, o consultarlos todos juntos con `.attrs`.

El texto que hay adentro de un tag (no el tag en sí, sino el contenido) es un objeto `NavigableString` — se comporta como un string normal, pero pertenece a la estructura de BeautifulSoup.

In [ ]:
tag_destacado = soup.b   # el <b id="destacado"> que vimos antes

print("atributo id con corchetes:", tag_destacado['id'])
print("atributo id con .get():", tag_destacado.get('id'))
print("todos los atributos:", tag_destacado.attrs)

tag_string = tag_destacado.string
print("contenido:", tag_string)
print("tipo:", type(tag_string))
print("como string normal de Python:", str(tag_string))

## `find()` y `find_all()`: filtrar el árbol

`find_all()` recorre todo el documento y devuelve una **lista** con todos los tags que coinciden con el filtro. `find()` devuelve directamente el **primer** resultado, sin envolverlo en una lista.

El filtro más simple es el nombre del tag, pero también se puede filtrar por atributos, por una lista de nombres, o por el texto exacto contenido en un elemento. Como `class` es una palabra reservada en Python, para filtrar por el atributo HTML `class` se usa `class_` (con guion bajo al final).

In [ ]:
# find(): el primer <td> del documento, sin lista
primer_td = soup.find('td')
print("find('td'):", primer_td)

# find_all(): TODOS los <td> del documento
todos_los_td = soup.find_all('td')
print("cantidad de <td> encontrados:", len(todos_los_td))

# Filtrar por una lista de nombres de tag
tags_tr_o_a = soup.find_all(name=["tr", "a"])
print("cantidad de <tr> o <a>:", len(tags_tr_o_a))

# Filtrar por atributo: el tag cuyo id es "tabla_productos"
tabla_por_id = soup.find(id="tabla_productos")
print("tabla encontrada por id:", tabla_por_id.name, tabla_por_id.get('id'))

# Filtrar por class_ (con guion bajo, porque class es palabra reservada en Python)
tabla_por_clase = soup.find("table", class_="catalogo")
print("tabla encontrada por class_:", tabla_por_clase.get('class'))

# Filtrar por href: todos los <a> que tengan href, sin importar el valor
links_con_href = soup.find_all('a', href=True)
print("links con href:", links_con_href)

# Filtrar por texto exacto contenido en un elemento
celdas_accesorios = soup.find_all(string="Accesorios")
print("celdas con el texto 'Accesorios':", celdas_accesorios)

## Extrayendo la tabla "a mano": `find_all('tr')` + `find_all('td')`

El patrón para recorrer cualquier tabla HTML es siempre el mismo: primero encontrás la tabla con `find('table')`, después recorrés sus filas con `find_all('tr')`, y dentro de cada fila recorrés sus celdas con `find_all('td')`.

Antes de escribir este tipo de código conviene mirar la tabla (acá, el `prettify()` de arriba) y anotar cuántas columnas tiene y en qué orden — así sabés de antemano qué índice de `cols[...]` corresponde a cada dato, en vez de adivinar. En nuestra tabla: columna 0 = nombre, columna 1 = categoría, columna 2 = precio, columna 3 = stock.

In [ ]:
tabla = soup.find('table')   # la única tabla del documento

filas = tabla.find_all('tr')
print("cantidad de filas (incluye encabezado):", len(filas))

productos = []
for fila in filas[1:]:   # saltamos la primera fila, que es el encabezado
    cols = fila.find_all('td')
    nombre = cols[0].get_text()     # .get_text() trae el texto aunque haya tags adentro (como el <a>)
    categoria = cols[1].get_text()
    precio = cols[2].string
    stock = cols[3].string
    productos.append((nombre, categoria, precio, stock))

for p in productos:
    print("{} | {} | {} | stock: {}".format(*p))

Fijate que usamos `.get_text()` para la columna de categoría en vez de `.string`: la celda del "Monitor 24 pulgadas" tiene un `<a>` adentro (`<td><a href="...">Pantallas</a></td>`), y `.string` solo funciona quando el tag tiene un único hijo de texto directo. `.get_text()` es más robusto porque junta todo el texto, sin importar cuántos tags haya anidados adentro.

## El atajo de pandas: `pd.read_html()`

Cuando lo único que necesitás es una tabla completa, no hace falta recorrerla celda por celda con BeautifulSoup — pandas trae `pd.read_html()`, que hace ese trabajo por vos y devuelve una **lista de DataFrames**, uno por cada tabla que encuentra.

`pd.read_html()` acepta una URL o un bloque de HTML. Desde pandas 2.1, la forma recomendada para pasarle un string de HTML (en vez de una URL o un archivo) es envolverlo con `io.StringIO(...)` — pasar el string directo funciona pero genera un aviso de advertencia (`FutureWarning`) en versiones recientes de pandas.

In [ ]:
import io
import pandas as pd

tablas = pd.read_html(io.StringIO(html_productos))
print("cantidad de tablas encontradas:", len(tablas))

df_productos = tablas[0]   # hay una sola tabla en la página, así que es la primera de la lista
df_productos

Compará el resultado: en una sola línea (`pd.read_html`) tenés un DataFrame listo para analizar con pandas, mientras que a mano con BeautifulSoup tuviste control total sobre qué texto tomar de cada celda (por ejemplo, decidir usar `.get_text()` en vez de `.string` para la celda con el link).

La regla práctica: usá `pd.read_html()` como primer intento cuando necesites una tabla completa — te ahorra código. Si la página no coopera (el contenido no está armado como `<table>` real, o viene con demasiada basura para limpiar), recurrís a BeautifulSoup para tener control total con `find_all('tr')` / `find_all('td')`.

## Resumen de la clase

- El web scraping extrae datos directamente del HTML de una página cuando no existe una API que los entregue de forma estructurada.
- `BeautifulSoup(html, 'html.parser')` convierte un string de HTML en un objeto navegable; `prettify()` lo muestra reindentado según su jerarquía.
- Cada elemento del HTML es un objeto `Tag`; `soup.<tag>` devuelve el primero que aparece en el documento.
- Se navega el árbol con `.parent` (sube), `.next_sibling` (al costado) y accesos directos como `soup.h3` (baja).
- Los atributos de un tag se acceden como un diccionario (`tag['id']`, `tag.get('id')`, `tag.attrs`); el texto interno es un `NavigableString`, similar a un string común.
- `find_all()` devuelve **todos** los elementos que cumplen un filtro (por nombre, atributos, `class_`, `href`, texto); `find()` devuelve solo el **primero**, sin lista.
- Una tabla se recorre siempre igual: `find('table')` para ubicarla, `find_all('tr')` para las filas, y `find_all('td')` dentro de cada fila para las celdas. `.get_text()` es más robusto que `.string` cuando una celda tiene tags anidados adentro.
- `pd.read_html(io.StringIO(html))` es el atajo de pandas: convierte directamente las tablas de un HTML en una lista de DataFrames, en una sola línea.

**Próxima clase:** vas a seguir combinando estas herramientas con proyectos que integran APIs, pandas y scraping para construir datasets propios.